In [15]:
from functools import partial

import astroplan as ap
from astropy.coordinates import EarthLocation
import astropy.units as u

from astropaul.database import html_path
import astropaul.targetlistcreator as tlc
import astropaul.html as html
import astropaul.phase as ph
import astropaul.priority as pr

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
name = "Fan Mountain SIDE 2026 Fall"
html_dir = html_path() / name
html.clear_directory(html_dir)

session = tlc.ObservingSession(
    ap.Observer(EarthLocation(lat="37:52:41.35", lon="-78:41:34.92", height=566 * u.m), timezone="utc", name="Fan Mountain")
)

session.add_day_range("2026-08-30", "2026-12-31")

phase_event_defs = [
    ph.PhaseEventDef("Not in Eclipse", partial(ph.calc_time_of_gress, ingress=False)),
    ph.PhaseEventDef("Eclipse", partial(ph.calc_time_of_gress, ingress=True)),
]

fan_targets = [
    "TIC 63459761",
    "TIC 344541836",
    "TIC 418699570",
]

min_altitude = 35 * u.deg

all_targets = tlc.TargetList.load()


creator = tlc.TargetListCreator(name=name, phase_event_defs=phase_event_defs)
creator.steps = [
    partial(tlc.filter_targets, criteria=lambda df: (df["Target Name"].isin(fan_targets))),
    partial(
        tlc.add_observability,
        observing_session=session,
        calc_moon_distance=True,
        observability_threshold=(min_altitude, 80 * u.deg),
    ),
    partial(tlc.filter_targets, criteria=lambda df: df["Observable Any Night"]),
    partial(
        tlc.add_phase_events,
        observing_session=session,
        phase_event_defs=phase_event_defs,
        event_types=["Mid Eclipse", "Eclipse"],
    ),
]
tl = creator.calculate(initial_list=all_targets, verbose=False)
tl.name = name

print(tl.summarize())

/Users/paul/Library/CloudStorage/Dropbox/Astro/astropaul/astropaul/targetlistcreator/observability.py:95: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  answer.target_list = answer.target_list.assign(**new_cols)
/Users/paul/Library/CloudStorage/Dropbox/Astro/astropaul/astropaul/targetlistcreator/observability.py:95: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  answer.target_list = answer.target_list.assign(**new_cols)
/Users/paul/Library/CloudStorage/Dropbox/Astro/astropaul/astropaul/targetlistcreator/observability.py:95: Perfor

Name: Fan Mountain SIDE 2026 Fall
Criteria:
  Target list loaded from file: TargetList_2026-09-05_12h41m38s.tl (750 targets)
  lambda df: (df["Target Name"].isin(['TIC 63459761', 'TIC 344541836', 'TIC 418699570'])) (3 targets)
  Observability calculated at Fan Mountain in 15.0 min intervals from 2026-08-29 to 2026-12-31
    AltitudeConstraint: {'min': np.float64(35.0), 'max': np.float64(80.0), 'boolean_constraint': True}
  lambda df: df["Observable Any Night"] (3 targets)
  
3 targets:
     3 QuadEB
Column Count (primary, secondary):
    Target: (3, 4)
    RV Calibration Targets: (1, 2)
    List: (0, 20)
    Count: (8, 0)
    TESS Data: (4, 0)
    Gaia Bailer Jones: (1, 2)
    Observable: (5, 496)
Associated tables:
    3258 rows,   3 columns: Catalog Membership
    1188 rows,   2 columns: List Memberships
     894 rows,   7 columns: Ephemerides
     716 rows, 126 columns: TESS
     318 rows, 105 columns: Gaia DR3
      49 rows,  16 columns: WDS
     511 rows,  12 columns: DSSI Observa

In [42]:
illumination_categories = [
    ((0.0, 0.4), "Dark"),
    ((0.4, 0.7), "Gray"),
    ((0.7, 1.0), "Bright"),
]

distance_categories = {
    "Dark": [
        ((0, 180), 1),
    ],
    "Gray": [
        ((0, 5), 0.1),
        ((5, 15), 0.85),
        ((15, 180), 1),
    ],
    "Bright": [
        ((0, 15), 0.25),
        ((15, 30), 0.75),
        ((30, 180), 1),
    ],
}

altitude_categories = [
    ((-90, min_altitude.value), 0),
    ((min_altitude.value, 45), 0.95),
    ((45, 90), 1),
]

phase_scores = {
    "Not in Eclipse|Not in Eclipse": 0.0,
    "Not in Eclipse|Eclipse": 1.0,
    "Eclipse|Not in Eclipse": 1.0,
    "Eclipse|Eclipse": 0.5,
}

pl = pr.PriorityList(tl, session, interval=30 * u.min)
pr.calculate_moon_priority(pl, illumination_categories=illumination_categories, dist_categories=distance_categories)
pr.calculate_altitude_priority(pl, altitude_categories=altitude_categories)
pr.calculate_phase_priority(pl, phase_event_defs, phase_scores)
pr.calculate_overall_priority(pl)
pr.aggregate_target_priorities(pl, skip_column_threshold=0.0)
pl.categorize_priorities(bins=[0.00, 0.20, 0.40, 0.6, 1.00], labels=["", "*", "* *", "* * *"])
html.render_observing_pages(tl, pl, {}, html_dir, target_pages="Local")

 '2026-08-30T01:00:00.000000000' '2026-08-30T01:30:00.000000000'
 '2026-08-30T02:00:00.000000000' '2026-08-30T02:30:00.000000000'
 '2026-08-30T03:00:00.000000000' '2026-08-30T03:30:00.000000000'
 '2026-08-30T04:00:00.000000000' '2026-08-30T04:30:00.000000000'
 '2026-08-30T05:00:00.000000000' '2026-08-30T05:30:00.000000000'
 '2026-08-30T06:00:00.000000000' '2026-08-30T06:30:00.000000000'
 '2026-08-30T07:00:00.000000000' '2026-08-30T07:30:00.000000000'
 '2026-08-30T08:00:00.000000000' '2026-08-30T08:30:00.000000000'
 '2026-08-30T09:00:00.000000000' '2026-08-30T09:30:00.000000000'
 '2026-08-30T10:00:00.000000000'], obsgeoloc=[(-937292.35097461, -4951222.22949816, 3897754.60058446),
 (-281104.51416402, -5032635.27750711, 3896045.66954202),
 ( 380094.22634898, -5027464.87586409, 3894320.83225029),
 (1034928.6339855 , -4935799.97589192, 3892609.76261322),
 (1672132.96389205, -4759217.57679251, 3890941.89769834),
 (2280744.7783627 , -4500755.5950162 , 3889345.93130403),
 (2850293.54417724, -4